# 검색어 만들기

그래프에서 **서비스 대상 후보 1,067개 모두**의 구글 뉴스 RSS 검색어를 미리 만든다(`graphDB.md` 6.5). 구글 뉴스 RSS 검색(프로젝트 루트의 `news_search/` 폴더)은 이 노트북이 내보낸 표에서 **보유 종목의 검색어만** 골라 검색한다.

- **종목 유형**: 업종을 6개 유형(제품·공급망 / 의약·바이오 / 브랜드·소비자 / 수주 / 원자재·가격 / 금융·거시)으로 묶어 기본 유형으로 쓰고, 테마로 보정한다(업종이 실제 사업과 다를 때).
- **검색어 종류**(모든 유형 공통): 자기 회사, 구체 제품, 브랜드, 해외 상대, 원자재·지표, 정책, 테마 공통. 종류마다 검색어 모양(주제만 / 회사 + 주제 / 이름만)이 정해져 있고, 유형은 종류마다 검색어를 몇 개씩 줄지만 정한다.
- **연결이 적은 종목**(미국 뉴스에 나온 검색 재료가 적은 중소형주)은 검색어를 테마 쪽으로 더 준다. 테마 검색어는 테마마다 한 번 만들어 같은 테마 종목이 함께 쓴다.
- 그래프 노드 ID는 빌드마다 바뀔 수 있으므로 결과는 **종목코드 기준**으로 내보낸다.
- 지금은 1단계(유형 대응표, 종목 유형, 연결이 적은 종목), 2단계(검색어 후보, 구글 뉴스 RSS 시험 목록), **전체 검색어 표**(12절, 2026-11-17)까지 있다. 밤사이 수집은 아직 시험 목록 사본을 쓰고, 검색어를 줄이는 규칙을 정한 뒤 전체 표로 바꾼다.

**입력**

| 파일 | 쓰는 것 |
|---|---|
| `source/graph/neo4j/nodes_*.csv`, `rels_*.csv` | `graph_db`가 만든 노드·관계 표(기업의 업종·테마, 제품·브랜드·해외 상대, 미국 뉴스 기사 수) |

**출력** (`source/search/`, 내용이 바뀔 때만 쓴다)

| 파일 | 한 줄 |
|---|---|
| `company_types.csv` | 서비스 대상 후보 하나: 업종, 테마, 기본 유형, 보조 유형, 정한 이유, 미국 뉴스에 나온 검색 재료 수, 연결이 적은지 |
| `base_terms.csv` | 업종·테마 하나: 기본 검색어(LLM, 9절) |
| `company_base_terms.csv` | 업종·테마 기본 검색어를 못 받는 종목 하나: 회사 기본 검색어(LLM, 10절) |
| `llm/` | 기본 검색어 LLM 원본(`raw/`), 실행 기록(`runs.csv`), 스키마 |
| `test_queries.csv` | 구글 뉴스 RSS 시험 목록: 시험 종목, 검색어 종류, 검색어, 모양, 검색 문자열, 근거 노드 |
| `queries_all.csv` | **전체 검색어 표**: 서비스 대상 후보 전체의 종목코드, 종목명, 검색어 종류, 검색어, 모양, 검색 문자열, 근거 노드(12절) |

**한눈에**

| | |
|---|---|
| 처리 순서에서 | `graph_db` → **`search_terms`** → 구글 뉴스 RSS 검색(`news_search/`) |
| 외부 요청·비용 | 업종·테마·회사 기본 검색어는 Codex 구독(요청 4건, 비용 없음, `RUN = True`일 때만). 구글 뉴스 RSS 검색은 이 노트북이 아니라 `news_search` 폴더에서 한다 |
| 다시 실행 | 언제든 된다. 그래프를 다시 만들면 이 노트북도 다시 돌린다 |
| 처리 순서 | 설정 → 유형 대응표 → 그래프 표 읽기 → 종목 유형 → 검색 재료와 연결이 적은 종목 → 저장 → 확인 → 검색어 후보 → 업종·테마 기본 검색어(Codex) → 회사 기본 검색어(Codex) → 시험 목록 → 전체 검색어 표 |

## 1. 설정

In [1]:
# ── 설정 ──
import json
from collections import Counter
from pathlib import Path

import pandas as pd

pd.set_option('display.max_colwidth', 80)
pd.set_option('display.width', 200)

ROOT = Path.cwd()
assert ROOT.name == 'graph_db', 'graph_db 폴더에서 실행해야 한다.'
NEO = ROOT / 'source' / 'graph' / 'neo4j'       # graph_db가 만든 노드·관계 표
OUT = ROOT / 'source' / 'search'
OUT.mkdir(parents=True, exist_ok=True)
WEAK_MAX = 1          # 미국 뉴스에 나온 검색 재료가 이 수 이하면 연결이 적은 종목(구글 뉴스 RSS 시험으로 조정)
BRAND_SIGNAL = 5      # 업종과 관계없이 브랜드가 이만큼 이상이면 브랜드·소비자형을 보조 유형으로 더한다
ORDER_SIGNAL = 5      # 수주 관계가 이만큼 이상이면 수주형을 보조 유형으로 더한다
loads = lambda s: json.loads(s) if isinstance(s, str) and s.startswith('[') else []


def save_csv(df, path):
    """내용이 바뀔 때만 쓴다(iCloud 사본 방지)."""
    text = df.to_csv(index=False)
    if path.exists() and path.read_text(encoding='utf-8-sig') == text:
        return '그대로'
    path.write_text(text, encoding='utf-8-sig')
    return '씀'

## 2. 유형 대응표

**업종 → 유형**(WI26 26개 모두)과 **테마 → 유형**(토스 상위 태그 125개 모두). 사업 내용을 말해 주지 않는 테마(지주사, 무역, 유통 등)는 `중립`으로 두고 유형 판단에 쓰지 않는다.

| 유형 | 뉴스가 종목으로 이어지는 고리 | 우선할 검색어 종류 |
|---|---|---|
| 제품·공급망 | 부품·소재·장비, 해외 고객사·경쟁사 | 구체 제품, 해외 상대 |
| 의약·바이오 | 약효군·적응증, 라이선스 파트너, FDA | 구체 제품(약효군), 해외 상대, 경쟁 약 브랜드 |
| 브랜드·소비자 | 자사 브랜드, 소비 흐름, 해외 유통 채널 | 브랜드, 구체 제품 |
| 수주 | 발주 분야, 발주처·국가, 정책 | 구체 제품(발주 분야), 해외 상대, 정책 |
| 원자재·가격 | 원자재·운임 가격, 무역 정책 | 원자재·지표, 정책 |
| 금융·거시 | 금리·거시(매일 밤 프롬프트의 "시장 전체"가 맡음) | 자기 회사 |

In [2]:
# ── 유형 대응표: 업종 → 유형, 테마 → 유형(중립은 유형 판단에 쓰지 않는다) ──
TYPES = ['제품·공급망', '의약·바이오', '브랜드·소비자', '수주', '원자재·가격', '금융·거시']
SECTOR_TYPE = {
    '반도체': '제품·공급망', '자동차': '제품·공급망', '화학': '제품·공급망', '기계': '제품·공급망', 'IT하드웨어': '제품·공급망',
    'IT가전': '제품·공급망', '디스플레이': '제품·공급망', '소프트웨어': '제품·공급망',       # 게임사는 게임 테마로 브랜드·소비자형이 된다
    '건강관리': '의약·바이오',
    '화장품,의류': '브랜드·소비자', '필수소비재': '브랜드·소비자', '미디어,교육': '브랜드·소비자', '소매(유통)': '브랜드·소비자', '호텔,레저': '브랜드·소비자',
    '건설': '수주', '상사,자본재': '수주', '조선': '수주',
    '비철금속': '원자재·가격', '철강': '원자재·가격', '운송': '원자재·가격', '에너지': '원자재·가격', '유틸리티': '원자재·가격',
    '증권': '금융·거시', '은행': '금융·거시', '보험': '금융·거시', '통신서비스': '금융·거시',
}
THEME_GROUPS = {
    '제품·공급망': ['반도체', '디스플레이', '이차전지', '전자부품', '스마트폰', '자동차', '자동차 전장', '전기차', '산업용 기계', '로봇', '통신장비',
                 '전기설비', '컴퓨터·주변기기', 'CCTV', 'LED', 'ESS', '보안', '소프트웨어', '클라우드 컴퓨팅', '인공지능', 'IT솔루션구축',
                 '화학', '산업용 가스', '페인트', '수소에너지', '탄소저감', '신재생에너지', '드론', '농업용 기계', '오토바이', '방송콘텐츠장비',
                 '금융기기', '쇼핑몰구축'],
    '의약·바이오': ['제약', '바이오', '의료기기', '의료서비스', '대마초'],
    '브랜드·소비자': ['음식료', '화장품', '패션·섬유', '가전', '생활용품', '건강기능식품', '마스크', '가구', '그릇', '담배', '주류', '음식점브랜드',
                  '게임', '디지털 콘텐츠', '엔터테인먼트', '캐릭터', '교육', '출판', '광고', '방송', '유료방송', '영화관', '소셜 미디어',
                  '인터넷 플랫폼', '구인구직', '여행·관광', '골프', '여가생활', '카지노', '렌터카', '자전거'],
    '수주': ['건설', '방산', '조선', '원자력발전', '우주항공', '철도', '시멘트', '마루', '욕실', '창호', '수자원'],
    '원자재·가격': ['철강', '비철금속', '금속', '석유·가스', '화력발전', '지역난방', '전력에너지', '물류·운송', '항공사', '버스', '참치원양어선',
                 '사료', '비료와 농약', '농산물종자', '농업', '식품포장재', '골판지', '종이', '백판지', '목재', '라면포장지', '우유팩', '주정', '폐기물처리'],
    '금융·거시': ['증권', '보험', '은행', '금융그룹', '카드', '부동산신탁', '부동산임대개발', '리츠', '벤처캐피탈', '핀테크', '신용평가', '금융',
               '금융상품거래소', '디지털 자산', '통신서비스'],
    '중립': ['지주사', '무역', '유통', 'IT', '경영지원', '컨택센터'],   # 유통: 소비재 유통과 부품·원료 유통(반도체 유통상 등)이 섞여 있다
}
THEME_TYPE = {t: g for g, ts in THEME_GROUPS.items() for t in ts}
assert sum(len(v) for v in THEME_GROUPS.values()) == len(THEME_TYPE), '두 유형에 든 테마가 있다'

## 3. 그래프 표 읽기

In [3]:
# ── 그래프 표 읽기: 기업, 제품, 계층, 브랜드 → 제품, 기업의 관계 ──
CO = pd.read_csv(NEO / 'nodes_Company.csv', dtype=str).fillna('')
TH = pd.read_csv(NEO / 'nodes_Theme.csv', dtype=str).fillna('')
SE = pd.read_csv(NEO / 'nodes_Sector.csv', dtype=str).fillna('')
PRODUCT = pd.read_csv(NEO / 'nodes_Product.csv', dtype=str).fillna('').set_index('id')
missing_sec, missing_th = set(SE['name']) - set(SECTOR_TYPE), set(TH['name']) - set(THEME_TYPE)
assert not missing_sec and not missing_th, (missing_sec, missing_th)
T = CO[CO['target'] == 'True'].copy()
T['themes'] = T['themes'].map(loads)
co_news = pd.to_numeric(CO.set_index('id')['n_news'], errors='coerce').fillna(0)
co_country = CO.set_index('id')['country']
p_news = pd.to_numeric(PRODUCT['n_news'], errors='coerce').fillna(0)
up = pd.read_csv(NEO / 'rels_SUBTYPE_OF.csv', dtype=str).set_index('start')['end'].to_dict()
brand_product = pd.read_csv(NEO / 'rels_OF_PRODUCT.csv', dtype=str).groupby('start')['end'].agg(list).to_dict()
SKIP = {'IN_SECTOR', 'HAS_THEME', 'IN_GROUP', 'SUBTYPE_OF', 'OF_PRODUCT'}
REL = pd.concat([pd.read_csv(f, dtype=str, usecols=['start', 'end']).assign(type=f.stem[5:]) for f in NEO.glob('rels_*.csv') if f.stem[5:] not in SKIP])
mine = REL[REL['start'].isin(T['id']) | REL['end'].isin(T['id'])]
mine = pd.concat([mine.assign(me=mine['start'], other=mine['end']), mine.assign(me=mine['end'], other=mine['start'])])
mine = mine[mine['me'].isin(T['id'])]
print(f"서비스 대상 후보 {len(T):,}개 / 업종 {len(SE)}개·테마 {len(TH)}개 모두 유형이 있다 / 후보가 걸린 관계 {len(mine):,}줄")

서비스 대상 후보 1,067개 / 업종 26개·테마 125개 모두 유형이 있다 / 후보가 걸린 관계 79,225줄


## 4. 종목 유형

1. **기본 유형**은 업종의 유형이다.
2. 테마(중립 제외)의 유형이 **모두 한 유형**이고 업종 유형과 다르면, 기본 유형을 테마 유형으로 바꾼다(업종이 실제 사업과 다른 경우).
3. 그 밖에 테마가 가리키는 다른 유형은 **보조 유형**으로 더한다(삼성전자: 반도체 → 제품·공급망 + 스마트폰·가전 테마 → 브랜드·소비자).
4. 그래프 연결로도 보조 유형을 더한다: 브랜드가 `BRAND_SIGNAL`개 이상이면 브랜드·소비자, 수주 관계가 `ORDER_SIGNAL`개 이상이면 수주.

In [4]:
# ── 종목 유형: 업종 → 기본 유형, 테마로 바꾸거나 보조 유형을 더하고, 그래프 연결로 보조 유형을 더한다 ──
n_brand = mine[mine['type'] == 'HAS_BRAND'].groupby('me').size()
n_order = mine[mine['type'] == 'WINS_ORDER'].groupby('me').size()
rows = []
for r in T.itertuples():
    sec_type = SECTOR_TYPE[r.sector]
    th_types = [THEME_TYPE[t] for t in r.themes if THEME_TYPE[t] != '중립']
    base, why = sec_type, [f'업종 {r.sector}']
    if th_types and len(set(th_types)) == 1 and th_types[0] != sec_type:
        base = th_types[0]
        why.append(f"테마({', '.join(t for t in r.themes if THEME_TYPE[t] != '중립')})가 모두 {base} → 기본 유형을 바꿈")
    extra = list(dict.fromkeys(t for t in th_types if t != base))
    if extra:
        why.append(f"테마로 보조 {', '.join(extra)}")
    if n_brand.get(r.id, 0) >= BRAND_SIGNAL and base != '브랜드·소비자' and '브랜드·소비자' not in extra:
        extra.append('브랜드·소비자')
        why.append(f'브랜드 {n_brand[r.id]}개 → 보조 브랜드·소비자')
    if n_order.get(r.id, 0) >= ORDER_SIGNAL and base != '수주' and '수주' not in extra:
        extra.append('수주')
        why.append(f'수주 관계 {n_order[r.id]}개 → 보조 수주')
    rows.append(dict(stock_code=r.stock_code, name=r.name, sector=r.sector, themes=' | '.join(r.themes), sector_type=sec_type,
                     base_type=base, extra_types=' | '.join(extra), reason=' / '.join(why)))
CT = pd.DataFrame(rows)
print('기본 유형', CT['base_type'].value_counts().to_dict())
print(f"테마로 기본 유형을 바꾼 종목 {(CT['base_type'] != CT['sector_type']).sum()}개 / 보조 유형이 있는 종목 {(CT['extra_types'] != '').sum()}개")

기본 유형 {'제품·공급망': 405, '브랜드·소비자': 205, '원자재·가격': 160, '수주': 119, '의약·바이오': 113, '금융·거시': 65}
테마로 기본 유형을 바꾼 종목 85개 / 보조 유형이 있는 종목 429개


## 5. 검색 재료와 연결이 적은 종목

**미국 뉴스에 나온 적 있는 검색 재료**를 종목마다 센다.

- 그 종목이 만드는 제품, 또는 그 위 계층 제품(2단계까지) 중 미국 뉴스 기사 수가 1 이상인 것.
- 그 종목 브랜드가 가리키는 제품 중 미국 뉴스에 나온 것.
- 그 종목과 관계(그룹 소속 제외)가 있는 해외 기업 중 미국 뉴스에 나온 것.

재료가 `WEAK_MAX`개 이하면 **연결이 적은 종목**이다. 이런 종목은 다음 단계에서 검색어를 테마 쪽으로 더 받는다. 지금 쌓인 기사는 2,471건뿐이라 기사가 늘면 수가 줄고, 기준은 구글 뉴스 RSS 시험으로 조정한다.

In [5]:
# ── 검색 재료: 미국 뉴스에 나온 제품(계층 위 포함)·브랜드의 제품·해외 상대 ──
def ancestors(p, depth=2):
    out = [p]
    while p in up and len(out) <= depth:
        p = up[p]
        out.append(p)
    return out


material = {}
for me, g in mine.groupby('me'):
    prods = {a for p in g.loc[g['type'] == 'PRODUCES', 'other'] for a in ancestors(p) if p_news.get(a, 0) > 0}
    brand_p = {p for b in g.loc[g['type'] == 'HAS_BRAND', 'other'] for p in brand_product.get(b, []) if p_news.get(p, 0) > 0}
    partners = {x for x in g['other'] if x in co_news.index and co_news[x] > 0 and co_country.get(x, '') not in ('', '대한민국')}
    material[me] = (len(prods), len(brand_p), len(partners))
id_of = dict(zip(T['stock_code'], T['id']))
CT[['m_products', 'm_brand_products', 'm_partners']] = [material.get(id_of[c], (0, 0, 0)) for c in CT['stock_code']]
CT['n_material'] = CT[['m_products', 'm_brand_products', 'm_partners']].sum(axis=1)
CT['weak'] = CT['n_material'] <= WEAK_MAX
print(f"연결이 적은 종목(재료 {WEAK_MAX}개 이하) {CT['weak'].sum()}개 / {len(CT):,}개")
display(CT.groupby('base_type').agg(종목=('stock_code', 'size'), 연결이_적은_종목=('weak', 'sum'), 재료_중앙값=('n_material', 'median'))
        .assign(비율=lambda d: (d['연결이_적은_종목'] / d['종목']).round(2)))

연결이 적은 종목(재료 1개 이하) 661개 / 1,067개

,종목,연결이_적은_종목,재료_중앙값,비율
base_type,,,,
금융·거시,65,55,0.0,0.85
브랜드·소비자,205,146,0.0,0.71
수주,119,70,1.0,0.59
원자재·가격,160,78,2.0,0.49
의약·바이오,113,67,1.0,0.59
제품·공급망,405,245,1.0,0.60


## 6. 저장

In [6]:
# ── 저장: 내용이 바뀔 때만 쓴다 ──
print('company_types.csv', save_csv(CT.sort_values('stock_code'), OUT / 'company_types.csv'))

company_types.csv 그대로


## 7. 확인

- 테마로 기본 유형을 바꾼 종목과 보조 유형이 붙은 대표 종목(삼성전자, 한화에어로스페이스 등)이 맞는지 본다.
- 연결이 적은 종목의 예.

In [7]:
# ── 확인: 기본 유형을 바꾼 종목, 대표 종목, 연결이 적은 종목 예 ──
cols = ['name', 'sector', 'themes', 'base_type', 'extra_types', 'n_material', 'weak']
print('■ 테마로 기본 유형을 바꾼 종목(무작위 15개)')
changed = CT[CT['base_type'] != CT['sector_type']]
display(changed.sample(min(15, len(changed)), random_state=1)[cols])
print('■ 대표 종목')
display(CT[CT['name'].isin(['삼성전자', 'SK하이닉스', '한화에어로스페이스', '현대차', '농심', '셀트리온', '펩트론', 'HD한국조선해양', 'POSCO홀딩스', 'KB금융', '하이브', '엔씨소프트', '크래프톤'])][cols])
print('■ 연결이 적은 종목(무작위 12개)')
display(CT[CT['weak']].sample(12, random_state=2)[cols])

■ 테마로 기본 유형을 바꾼 종목(무작위 15개)


,name,sector,themes,base_type,extra_types,n_material,weak
501,NICE평가정보,"상사,자본재",신용평가,금융·거시,브랜드·소비자,0,True
632,SOOP,소프트웨어,디지털 콘텐츠,브랜드·소비자,,2,False
369,경동인베스트,비철금속,지주사 | 건설,수주,,3,False
581,코스맥스비티아이,건강관리,지주사 | 건강기능식품,브랜드·소비자,,4,False
458,태광,기계,건설 | 조선,수주,,0,True
1007,에코프로에이치엔,"상사,자본재",탄소저감,제품·공급망,,0,True
795,아이마켓코리아,"상사,자본재",유통 | 제약 | 의료기기,의약·바이오,,4,False
915,크래프톤,소프트웨어,게임,브랜드·소비자,,3,False
1062,HS효성,"상사,자본재",지주사 | 물류·운송 | IT,원자재·가격,브랜드·소비자,3,False
768,우진,기계,원자력발전,수주,,1,True


■ 대표 종목


,name,sector,themes,base_type,extra_types,n_material,weak
30,SK하이닉스,반도체,반도체,제품·공급망,,11,False
182,농심,필수소비재,음식료,브랜드·소비자,,0,True
216,현대차,자동차,자동차 | 전기차 | 금융 | 철도 | 방산 | 건설,제품·공급망,금융·거시 | 수주 | 브랜드·소비자,7,False
220,POSCO홀딩스,철강,지주사 | 철강 | 무역 | 화력발전 | 이차전지 | 건설,원자재·가격,제품·공급망 | 수주,10,False
235,삼성전자,반도체,반도체 | 스마트폰 | 컴퓨터·주변기기 | 가전 | 디스플레이 | 통신장비 | 자동차 전장,제품·공급망,브랜드·소비자 | 수주,24,False
317,HD한국조선해양,조선,지주사 | 조선 | 건설 | 신재생에너지,수주,제품·공급망,16,False
371,한화에어로스페이스,"상사,자본재",방산 | 우주항공 | CCTV | IT솔루션구축 | 산업용 기계,수주,제품·공급망 | 브랜드·소비자,9,False
635,셀트리온,건강관리,바이오 | 제약,의약·바이오,브랜드·소비자,4,False
701,펩트론,건강관리,바이오 | 제약,의약·바이오,,3,False
766,KB금융,은행,금융그룹 | 은행 | 보험 | 증권 | 카드 | 부동산신탁 | 벤처캐피탈 | IT솔루션구축,금융·거시,제품·공급망,2,False


■ 연결이 적은 종목(무작위 12개)


,name,sector,themes,base_type,extra_types,n_material,weak
449,아시아나항공,운송,항공사 | IT솔루션구축 | 여행·관광 | 골프,원자재·가격,제품·공급망 | 브랜드·소비자,1,True
506,교보증권,증권,증권,금융·거시,,0,True
323,명신산업,자동차,자동차 | 전기차,제품·공급망,,0,True
440,한온시스템,자동차,자동차 | 전기차,제품·공급망,,1,True
335,화천기계,기계,산업용 기계 | 자동차,제품·공급망,,1,True
552,NC,소프트웨어,게임,브랜드·소비자,,0,True
300,삼성전기,IT하드웨어,스마트폰 | 전자부품 | 반도체,제품·공급망,,1,True
393,한솔케미칼,화학,화학 | 이차전지 | 전자부품 | 생활용품,제품·공급망,브랜드·소비자,1,True
994,SK아이이테크놀로지,IT가전,이차전지,제품·공급망,,0,True
68,BYC,"화장품,의류",패션·섬유 | 부동산임대개발,브랜드·소비자,금융·거시,0,True


## 8. 검색어 후보

종목마다 검색어 종류별 후보를 그래프에서 뽑는다. 순위는 **미국 뉴스 기사 수**(그래프 노드의 `n_news`)를 먼저 본다.

| 종류 | 뽑는 곳 | 순위·거르기 |
|---|---|---|
| 자기 회사 | 기업 노드의 영어 이름과 다른 이름(한글 없는 것)에서 법인 표기(Co., Ltd., Inc., Corp., Holdings 등)를 뗀 것 | 서로 다른 것 2개까지(긴 이름, 짧은 이름). 3자 이하 이름(LX, GKL)은 다른 뜻·회사와 겹쳐 쓰지 않고 긴 이름(Grand Korea Leisure, 짧은 것뿐이면 Holdings·Group을 남긴 LX Holdings)을 쓴다 |
| 구체 제품 | 그 종목이 직접 만드는 제품 2개(개발 코드명·모델명은 미국 뉴스에 나온 것만) + 위 계층 제품(2단계까지) 중 미국 뉴스에 나온 것 1개 | 미국 뉴스 기사 수, 그다음 만드는 국내 상장사가 적은 순. 일반적인 제품이거나 국내 상장사 10곳 이상이 만들면 **넓은 제품**으로 따로 1개 |
| 브랜드 | 그 종목의 브랜드(영어 이름에 한글 없는 것, 회사·그룹 이름·브랜드 아님 제외). 브랜드·소비자나 의약·바이오 유형이 아니면 미국 뉴스에 나온 것만(B2B 모델명 제외) | 미국 뉴스 기사 수 |
| 해외 상대 | 그 종목과 관계(지분·그룹 소속 제외)가 있는 해외 기업. 이름은 대표 영어 이름(짧은 별칭 MS는 모호해 쓰지 않음, 3자 이하 GM은 긴 이름 General Motors) | 미국 뉴스 기사 수. 관계에 제품이 적혀 있으면 "상대 + 제품" 모양도 만든다 |
| 원자재·지표 | 그 종목이 사는 것(구매·사용) | 미국 뉴스에 나온 것만, 넓은 것(철강 제품, 컴퓨터 장비) 제외 |
| 정책 | 그 종목의 제품·원자재를 규제하거나 영향을 주는 뉴스 정책 | 미국 뉴스 기사 수 |
| 테마 대표 | 같은 테마 종목들이 만드는 제품(위 계층 1단계 포함) 중 미국 뉴스에 나온 것, 3단어까지 | 테마 안 생산 종목 수 × 테마 집중도(그 제품을 만드는 후보 중 테마 종목의 비율) × log(1 + 기사 수). 테마 종목이 10개 이상이면 2개 이상이 만드는 것만, 집중도 0.4 이상만. 테마마다 3개. 집중도가 없으면 지주사·대기업이 만드는 기사 많은 제품(유조선, 원유)이 여러 테마에 올라오고, 일반어를 거른 뒤에는 화장품·음식료 테마에 GLP-1(집중도 0.18)이 올라왔다 |

**일반 영어 한 단어는 뺀다**(2차 시험: juice 140건, documentaries 159건, ACROSS 1,174건이 대부분 상관없는 기사). 영어 사전(WordNet, `product_dictionary`가 `source/nltk_data`에 받아 둔 것)에 있는 한 단어를 일반어로 본다. 약칭(대문자 4자 이하: HBM, LNG, DRAM), 원자재·지표(nickel, ethanol), 회사 이름(Apple, Shell)은 빼지 않는다. 빠진 자리는 다음 순위 후보가 채운다.

업종·테마 기본 검색어는 9절에서 LLM으로 만든다.

In [8]:
# ── 검색어 후보: 종목마다 종류별 후보(자기 회사, 구체·넓은 제품, 브랜드, 해외 상대, 원자재·지표, 정책, 테마 대표) ──
import math
import re

HANGUL = re.compile('[가-힣]')
CORP_RE = re.compile(r'(,?\s*\b(co|corp|corporation|inc|ltd|limited|company|holdings?|group|and)\b\.?)+[\s,.]*$', re.I)
CODE_LIKE = re.compile(r'^[A-Za-z]{1,5}[- ]?[A-Za-z]?\d[\w-]*$')        # 개발 코드명·모델명(ABP102, CT-G20, Raon-1)
SHORT_NAME = 3                                   # 3자 이하 회사 이름(LX, GKL, GM)은 다른 뜻·다른 회사와 겹쳐 긴 이름을 쓴다

# 일반 영어 한 단어(juice, drones, ACROSS)는 하루 수백 건씩 걸려 제외한다. 영어 사전(WordNet)에 있는 한 단어 = 일반어.
# 약칭(대문자 4자 이하: HBM, LNG, DRAM)과 원자재·지표, 회사 이름은 제외하지 않는다. WordNet은 product_dictionary가 받아 둔 것을 쓴다
import nltk
nltk.data.path.insert(0, str(ROOT / 'source' / 'nltk_data'))
try:
    from nltk.corpus import wordnet as wn
    wn.synsets('juice')
    HAS_WN = True
except LookupError:
    HAS_WN = False
    print('※ 알림: WordNet(source/nltk_data)이 없어 일반 영어 한 단어를 거르지 못한다. product_dictionary를 먼저 실행하면 받아진다.')


def generic(term):
    w = term.strip()
    return HAS_WN and ' ' not in w and not (w.isupper() and len(w) <= 4) and bool(wn.synsets(w.lower()))


class _Corp:
    """법인 표기를 끝에서 반복해 뗀다(Eli Lilly and Company → Eli Lilly, SAMSUNG ELECTRONICS CO., LTD. → SAMSUNG ELECTRONICS)."""
    def __init__(self, pattern):
        self.pattern = pattern

    def sub(self, _, s):
        prev = None
        while prev != s:
            prev, s = s, self.pattern.sub('', s).strip(' ,.')
        return s


CORP = _Corp(CORP_RE)
LIGHT_CORP = _Corp(re.compile(r'(,?\s*\b(co|corp|corporation|inc|ltd|limited|company)\b\.?)+[\s,.]*$', re.I))   # Holdings·Group은 남긴다(LX Holdings)
BROAD_MAKERS = 10
BRAND = pd.read_csv(NEO / 'nodes_Brand.csv', dtype=str).fillna('').set_index('id')
POLICY = pd.read_csv(NEO / 'nodes_Policy.csv', dtype=str).fillna('').set_index('id')
CO_I = CO.set_index('id')
num = lambda s: pd.to_numeric(s, errors='coerce').fillna(0)
p_makers, b_news, pol_news = num(PRODUCT['n_kr_makers']), num(BRAND['n_news']), num(POLICY['n_news'])
p_generic = PRODUCT['generic'] == 'True'
REL_P = pd.concat([pd.read_csv(f, dtype=str, usecols=['start', 'end', 'products']).assign(type=f.stem[5:]) for f in NEO.glob('rels_*.csv') if f.stem[5:] not in SKIP])
REL_P['products'] = REL_P['products'].map(loads)


def short_names(cid):
    """회사 영어 이름 후보: 법인 표기를 떼고 대소문자 무시로 한 번씩, 긴 것과 짧은 것 2개까지."""
    r = CO_I.loc[cid]
    names = [r['name_en']] + [x for x in loads(r['other_names']) if not HANGUL.search(x)]
    out = {}
    for x in names:
        y = CORP.sub('', x).strip(' ,.')
        if len(y) >= 2:
            out.setdefault(y.lower(), y)
    vals = sorted(out.values(), key=len)
    longs = [v for v in vals if len(v) > SHORT_NAME]
    if not longs:                                # 짧은 이름뿐이면 Holdings·Group을 남긴 이름(LX → LX Holdings)
        y = LIGHT_CORP.sub('', r['name_en']).strip(' ,.')
        longs = [y] if len(y) > SHORT_NAME else []
    vals = longs or vals                         # 그래도 없으면(TFE) 짧은 이름을 그대로 쓴다
    return list(dict.fromkeys([vals[-1], vals[0]])) if vals else []


def p_name(pid):
    return PRODUCT.at[pid, 'name_en'] if pid in PRODUCT.index else ''


def broad(pid):
    return p_generic.get(pid, False) or p_makers.get(pid, 0) >= BROAD_MAKERS


# 테마 대표: 테마 종목들이 만드는 제품(위 계층 1단계 포함) 중 미국 뉴스에 나온 것
theme_of = {r.id: r.themes for r in T.itertuples()}
prod_of = mine[mine['type'] == 'PRODUCES'].groupby('me')['other'].agg(set).to_dict()
# 점수 = 테마 안 생산 종목 수 × 테마 집중도(그 제품을 만드는 후보 중 테마 종목 비율) × log(1 + 기사 수). 테마 종목이 10개 이상이면 2개 이상이 만드는 것만
made = {c: {a for p in prod_of.get(c, ()) for a in ancestors(p, 1)} for c in theme_of}
all_cnt = Counter(a for ps in made.values() for a in ps)
THEME_MIN_SHARE = 0.4      # 테마 집중도가 이보다 낮으면 그 테마를 대표하지 않는다(화장품 → GLP-1 0.18, 이차전지 → crude oil 0.33)
THEME_MAX_WORDS = 3        # 제품 사전의 긴 이름(lithium iron phosphate cathode material)은 기사에 그대로 나오지 않는다
THEME_REP = {}
for th in TH['name']:
    members = [c for c, ts in theme_of.items() if th in ts]
    cnt = Counter(a for c in members for a in made[c] if p_news.get(a, 0) > 0 and not broad(a) and not generic(p_name(a))
                  and len(p_name(a).split()) <= THEME_MAX_WORDS)
    keep = [a for a in cnt if cnt[a] >= (2 if len(members) >= 10 else 1) and cnt[a] / all_cnt[a] >= THEME_MIN_SHARE]
    THEME_REP[th] = sorted(keep, key=lambda a: (-cnt[a] * (cnt[a] / all_cnt[a]) * math.log1p(p_news[a]), a))[:3]   # 동점은 ID 순서(실행마다 같게)


def candidates(cid):
    """종목 하나의 검색어 후보: [(종류, 검색어, 함께 쓸 말, 근거 노드, 미국 뉴스 기사 수)]."""
    g = mine[mine['me'] == cid]
    out = [('자기 회사', n, '', cid, int(co_news.get(cid, 0))) for n in short_names(cid)]
    direct = set(g.loc[g['type'] == 'PRODUCES', 'other'])
    prods = {a for p in direct for a in ancestors(p)}
    rank = lambda xs: sorted(xs, key=lambda a: (-p_news.get(a, 0), p_makers.get(a, 0), a))
    spec = [a for a in rank(direct) if not broad(a) and not generic(p_name(a)) and (p_news.get(a, 0) > 0 or not CODE_LIKE.match(p_name(a)))][:2]   # 직접 만드는 구체 제품 먼저(코드명은 미국 뉴스에 나온 것만)
    spec += [a for a in rank(prods - direct) if not broad(a) and not generic(p_name(a)) and p_news.get(a, 0) > 0 and a not in spec][:1]   # 위 계층은 미국 뉴스에 나온 것 하나
    out += [('구체 제품', p_name(a), '', a, int(p_news.get(a, 0))) for a in spec]
    out += [('넓은 제품', p_name(a), '', a, int(p_news.get(a, 0))) for a in [a for a in rank(prods) if broad(a) and not generic(p_name(a))][:1]]
    ct = CT.set_index('stock_code').loc[CO_I.at[cid, 'stock_code']]
    consumer = bool({'브랜드·소비자', '의약·바이오'} & ({ct['base_type']} | set(ct['extra_types'].split(' | '))))
    brands = [b for b in g.loc[g['type'] == 'HAS_BRAND', 'other'] if b in BRAND.index and BRAND.at[b, 'name_en'] and not HANGUL.search(BRAND.at[b, 'name_en'])
              and BRAND.at[b, 'brand_kind'] not in ('회사·그룹 이름', '브랜드 아님') and (consumer or b_news.get(b, 0) > 0) and not generic(BRAND.at[b, 'name_en'])]   # B2B 모델명은 미국 뉴스에 나온 것만
    out += [('브랜드', BRAND.at[b, 'name_en'], '', b, int(b_news.get(b, 0))) for b in sorted(set(brands), key=lambda b: (-b_news.get(b, 0), b))[:2]]
    rp = REL_P[((REL_P['start'] == cid) | (REL_P['end'] == cid)) & (REL_P['type'] != 'OWNS')]
    partners = {}
    for r in rp.itertuples():
        x = r.end if r.start == cid else r.start
        if x in CO_I.index and CO_I.at[x, 'country'] not in ('', '대한민국') and co_news.get(x, 0) > 0:
            partners.setdefault(x, set()).update(r.products)
    for x in sorted(partners, key=lambda x: (-co_news[x], x))[:2]:
        nm = CORP.sub('', CO_I.at[x, 'name_en']).strip(' ,.')                # 상대는 대표 영어 이름(짧은 별칭은 모호: MS)
        if len(nm) <= SHORT_NAME:                                              # 3자 이하(GM)는 긴 이름(General Motors)
            nm = (short_names(x) or [nm or CO_I.at[x, 'name']])[0]
        prod = next((p_name(p) for p in sorted(partners[x]) if p in PRODUCT.index), '')
        out.append(('해외 상대', nm, prod, x, int(co_news[x])))
    buys = [p for p in g.loc[g['type'] == 'BUYS', 'other'] if p_news.get(p, 0) > 0 and not broad(p)]
    out += [('원자재·지표', p_name(p), '', p, int(p_news[p])) for p in sorted(set(buys), key=lambda p: (-p_news[p], p))[:2]]
    targets = prods | set(buys)
    pol = REL_P[REL_P['start'].str.startswith('N:policy:') & REL_P['end'].isin(targets)]['start']
    out += [('정책', POLICY.at[x, 'name'], '', x, int(pol_news.get(x, 0))) for x in sorted(set(pol), key=lambda x: (-pol_news.get(x, 0), x))[:1]]
    seen = {o[3] for o in out}
    for th in [t for t in theme_of.get(cid, []) if THEME_TYPE[t] != '중립'][:2]:
        for a in THEME_REP[th][:2]:
            if a not in seen:
                seen.add(a)
                out.append(('테마 대표', p_name(a), th, a, int(p_news[a])))
    return out


print('테마 대표 검색어 예')
for t in ['반도체', '방산', '화장품', '이차전지', '게임', '은행', '조선', '음식료', '바이오', '철강', '엔터테인먼트', '카지노', '원자력발전', '자동차']:
    print(f'  {t}: {[p_name(a) for a in THEME_REP[t]]}')
print(f"테마 대표 검색어가 없는 테마 {sum(not v for v in THEME_REP.values())}개 / {len(THEME_REP)}개")
print('일반 영어 한 단어 판정 예:', {w: generic(w) for w in ['juice', 'drones', 'ACROSS', 'ramen', 'HBM', 'DRAM', 'K-beauty', 'Weverse']})
print('짧은 이름 → 긴 이름 예:', {n: short_names(id_of[c]) for n, c in CT.set_index('name')['stock_code'].items() if n in ('LX홀딩스', 'GKL', '티에프이', 'KB금융')})

테마 대표 검색어 예
  반도체: ['semiconductor chips', 'semiconductor testing services', 'high-bandwidth memory']
  방산: ['oil tankers', 'military communications systems', 'crude oil tankers']
  화장품: []
  이차전지: ['cathode active materials', 'electric vehicle batteries', 'battery cells']
  게임: ['online games']
  은행: ['payment cards', 'options trading']
  조선: ['oil tankers', 'container ships', 'oil product tankers']
  음식료: ['PET bottles']
  바이오: ['COVID-19 vaccine', 'influenza vaccine', 'therapeutic cancer vaccines']
  철강: ['reinforcing steel bar', 'metal scrap', 'galvanized steel sheet']
  엔터테인먼트: []
  카지노: []
  원자력발전: []
  자동차: ['hybrid electric vehicles', 'steel wire rod', 'vehicle braking systems']
테마 대표 검색어가 없는 테마 59개 / 125개
일반 영어 한 단어 판정 예: {'juice': True, 'drones': True, 'ACROSS': True, 'ramen': False, 'HBM': False, 'DRAM': False, 'K-beauty': False, 'Weverse': False}
짧은 이름 → 긴 이름 예: {'KB금융': ['KB Financial'], 'GKL': ['Grand Korea Leisure'], 'LX홀딩스': ['LX Holdings'], '티에프이': ['TFE']}


## 9. 업종·테마 기본 검색어 (Codex)

업종 26개와 테마(중립·종목 없는 테마 제외)마다 **산업·시장 수준의 미국 뉴스 검색어 3~5개**를 LLM으로 만든다. 회사 고유의 연결이 적은 종목은 이 검색어로 주로 기사를 받는다.

- 입력 한 줄: ID, 업종/테마, 이름, 종목 수, 주된 업종, 대표 종목(관계가 많은 순 6개, 영어 이름), 대표 제품(여러 종목이 만들고 그 묶음에 몰린 제품 8개), 미국 뉴스에 나온 제품(8절의 대표 검색어와 같은 방식).
- 지시문 `prompts/search_base_v4.md`(최종본만 남김, 버전별 차이는 아래 표): 미국 기사에 그대로 나오는 2단어 중심 표현(3단어까지), 서로 다른 측면(수요·공급·가격·정책·경쟁), 너무 넓은 말과 회사·제품 고유 이름은 쓰지 않음, 사업 내용을 말해 주지 않는 묶음은 비움.

| 지시문 | 결과 | 고친 것 |
|---|---|---|
| v1 | 139개 중 131개에 검색어, 8개 비움(상사,자본재, 금융, 리츠, 부동산신탁, 여가생활, 전력에너지, 방송콘텐츠장비, 버스). 우유팩 테마에 "power capacitors"(종목 삼영의 콘덴서 필름 사업을 따름) | - |
| v2 | 비운 것 3개(상사,자본재, 금융, 여가생활). 우유팩 → beverage cartons, liquid packaging board. 그러나 건설·가구·마루·욕실에 미국 국내 지표(housing starts)가 들어옴. v1과 같은 검색어 비율 평균 45%(실행마다 많이 흔들림) | 묶음 이름이 가리키는 사업을 기준으로, 대표 종목·제품은 넓은 이름을 좁히거나 모호한 이름을 이해하는 데만. 이름과 맞지 않는 종목의 다른 사업은 쓰지 않음(회사 검색어가 맡음). 종목 1~3개인 작은 묶음에 특히 주의. 이름이 분명한 사업(우유팩, 버스)은 비우지 않음 |
| v3 | 비운 것 6개(상사,자본재, 소매(유통), 금융, 부동산신탁, 부동산임대개발, 여가생활: 국내 내수 중심이라 미국 뉴스 경로가 약하다고 판단). 미국 국내 지표가 빠지고 경로가 분명해짐(건설 → LNG plant construction·Middle East megaprojects, 카지노 → Macau casino revenue·China outbound tourism, 은행 → dollar funding costs). 그러나 평균 2.73단어(3단어 385/532개)라 구글 뉴스 RSS 2차 시험(본문 확보 매체)에서 시험한 64개의 중앙값 0건, 66%가 0건 | 한국 종목에 이어지는 경로(한국 종목이 파는 것의 수요, 사고파는 것의 세계 가격, 미국·국제 정책, 세계 경쟁사)가 있는 뉴스만. 미국 국내만의 지표는 한국 종목이 미국에 그것을 팔 때만. 검색어를 중요한 순서로(시험·서비스는 앞의 것부터 쓴다) |
| v4 | 평균 1.99단어(2단어 493/507개). 비운 것 5개(소매(유통)에 duty-free sales가 생김). 3차 시험에서 시험한 65개의 중앙값 6건, 0건 17%(반도체 → DRAM prices·HBM demand·memory prices, 화장품,의류 → K-beauty, 미디어,교육 → K-pop). v4를 쓴다 | 검색은 따옴표로 표현 그대로 찾으므로 **2단어 기본, 꼭 필요할 때만 3단어, 4단어 이상 금지**. 핵심 명사 + 흔히 붙는 말(prices, demand, tariffs). 좋은 예·나쁜 예(artillery ammunition demand → artillery shells). 한 단어 일반어 금지 목록에 juice·animation 추가, 고유한 약칭(HBM, LNG)은 한 단어 허용 |
- **2026-11-17 다시 실행**: 그래프 표기를 통일하자(`graphDB.md` 6.2) 입력의 대표 종목 영어 이름 표기가 바뀌어("SAMSUNG ELECTRONICS" → "Samsung Electronics") v4를 같은 설정으로 다시 돌렸다. 평균 1.99단어, 비운 것 7개(v4 첫 실행의 5개 + 소매(유통), 폐기물처리. 첫 실행 때 소매(유통)에 있던 duty-free sales가 이번에는 없음). 앞의 2개(검색어 표에 쓰는 것)가 같은 묶음은 139개 중 60개, 검색어로는 268개 중 174개가 같다(반도체: DRAM prices, HBM demand → memory prices, HBM demand / 은행: Treasury yields, Fed policy → Treasury yields, dollar funding / 건설·방산은 그대로). 같은 설정이라도 실행마다 표현이 바뀌는 정도다.
- 실행: Codex 구독, 모델 `gpt-6.1-sol`, 추론 강도 high(요청 3건의 일회성 작업이라 가장 꼼꼼한 설정). 업종 1건, 테마 2건. 같은 지시문·입력으로 끝난 요청은 건너뛴다. `RUN = True`일 때만 보낸다.
- 원본은 `source/search/llm/raw/`, 실행 기록은 `source/search/llm/runs.csv`, 결과는 `source/search/base_terms.csv`.

In [9]:
# ── 기본 검색어 입력: 업종·테마마다 대표 종목·대표 제품·미국 뉴스에 나온 제품 ──
import datetime as dt
import hashlib
import os
import subprocess
import tempfile
import time
from concurrent.futures import ThreadPoolExecutor, as_completed

import jsonschema

BASE_PROMPT = 'search_base_v4'                     # 버전별 차이는 9절 표
PREV_PROMPT = 'search_base_v3'                     # 비교할 이전 버전
BASE_MODEL, BASE_EFFORT = 'gpt-6.1-sol', 'high'
RUN = False                                        # True로 바꿔야 Codex로 보낸다
LLM_DIR = OUT / 'llm'
degree = mine.groupby('me').size()
en_name = lambda c: CORP.sub('', CO_I.at[c, 'name_en']).strip(' ,.') or CO_I.at[c, 'name']


def group_products(members, need_news, k):
    # 묶음에 몰린 제품: 묶음 안 생산 종목 수 × 집중도(× log(1 + 기사 수)). 종목 10개 이상이면 2개 이상이 만드는 것만
    cnt = Counter(a for c in members for a in made.get(c, ()) if a in PRODUCT.index and not broad(a) and (p_news.get(a, 0) > 0 or not need_news))   # 제품군(PF:) 제외
    keep = [a for a in cnt if cnt[a] >= (2 if len(members) >= 10 else 1)]
    w = (lambda a: math.log1p(p_news[a])) if need_news else (lambda a: 1)
    return [p_name(a) for a in sorted(keep, key=lambda a: (-cnt[a] * (cnt[a] / all_cnt[a]) * w(a), a))[:k]]


GROUPS = []
for sec in sorted(SE['name']):
    GROUPS.append(('S:' + sec, '업종', sec, [id_of[c] for c in CT.loc[CT['sector'] == sec, 'stock_code']]))
for th in sorted(TH['name']):
    members = [c for c, ts in theme_of.items() if th in ts]
    if THEME_TYPE[th] != '중립' and members:
        GROUPS.append(('T:' + th, '테마', th, members))
lines = {}
for gid, kind, name, members in GROUPS:
    secs = Counter(CO_I.at[c, 'sector'] for c in members).most_common(2)
    reps = sorted(members, key=lambda c: -degree.get(c, 0))[:6]
    lines[gid] = ' | '.join([gid, kind, name, str(len(members)), ', '.join(f'{a} {b}' for a, b in secs),
                             ', '.join(en_name(c) for c in reps), ', '.join(group_products(members, False, 8)), ', '.join(group_products(members, True, 8))])
sec_ids = [g[0] for g in GROUPS if g[1] == '업종']
th_ids = [g[0] for g in GROUPS if g[1] == '테마']
half = (len(th_ids) + 1) // 2
BASE_INPUTS = {'base_sector': sec_ids, 'base_theme_1': th_ids[:half], 'base_theme_2': th_ids[half:]}
BASE_TEXTS = {k: '\n'.join(lines[g] for g in v) for k, v in BASE_INPUTS.items()}
print(f'업종 {len(sec_ids)}개, 테마 {len(th_ids)}개(중립·종목 없는 테마 제외) → 요청 {len(BASE_INPUTS)}건', {k: f'{len(v):,}자' for k, v in BASE_TEXTS.items()})
print(lines['T:방산'])
print(lines['S:반도체'])

업종 26개, 테마 113개(중립·종목 없는 테마 제외) → 요청 3건 {'base_sector': '10,259자', 'base_theme_1': '20,203자', 'base_theme_2': '21,961자'}
T:방산 | 테마 | 방산 | 33 | 상사,자본재 14, 자동차 6 | Hanwha, Hyundai Motor, Doosan, Hanwha Aerospace, HJ Shipbuilding & Construction, Hanwha Ocean | defense equipment components, military ammunition, military communications systems, military radar, armored vehicles, avionics, electro-optical satellite payloads, military electro-optical equipment | oil tankers, military communications systems, crude oil tankers, special-purpose vessels, LNG carriers, container ships, metal scrap, LPG carriers
S:반도체 | 업종 | 반도체 | 85 | 반도체 85 | Samsung Electronics, SK hynix, WONIK, SHINSUNG E&G, TSE, SK Square | semiconductor testing services, semiconductor test sockets, central chemical supply system, DRAM, semiconductor cleaning equipment, semiconductor gas equipment, semiconductor test pins, semiconductor test boards | DRAM, semiconductor testing services, high-bandwidth memory, NAND flash memory

In [10]:
# ── Codex 실행 도구와 기본 검색어 실행 ──
RUN_CWD = Path(tempfile.gettempdir()) / 'graphdb_llm_run'
RUN_CWD.mkdir(exist_ok=True)
ENV = {k: v for k, v in os.environ.items() if k not in ('ANTHROPIC_API_KEY', 'ANTHROPIC_AUTH_TOKEN', 'OPENAI_API_KEY')}   # 구독으로만 돈다
LIMIT_RE = re.compile(r'usage limit|rate limit|limit reached|quota|too many requests|429', re.I)
CODEX_SUFFIX = '\n\n## 결과 형식\n\n결과는 주어진 출력 스키마에 맞는 JSON 하나다. 파일을 읽거나 명령을 실행하지 말고, 아래 입력만 보고 답한다.'
RUNS_PATH = LLM_DIR / 'runs.csv'
RUN_COLUMNS = ['stage', 'item', 'input_hash', 'prompt_hash', 'runner', 'model', 'effort', 'run_utc', 'raw_file', 'status', 'error', 'duration_s',
               'input_tokens', 'cache_read_tokens', 'output_tokens', 'reasoning_tokens']
text_hash = lambda t: hashlib.sha256(t.encode('utf-8')).hexdigest()[:16]
read_runs = lambda: pd.read_csv(RUNS_PATH, dtype=str).reindex(columns=RUN_COLUMNS) if RUNS_PATH.exists() else pd.DataFrame(columns=RUN_COLUMNS)


def parse_result(resp):
    text = re.sub(r'^```(?:json)?\s*|\s*```$', '', (resp.get('last_message') or '').strip())
    return json.loads(text[text.find('{'):])


def run_codex(stage, item, prompt, schema_path, prompt_hash, text, model, effort):
    # Codex로 항목 하나를 돌리고 원본 응답을 저장한 뒤 실행 기록 한 줄을 돌려준다(product_llm의 같은 도구)
    now = dt.datetime.now(dt.timezone.utc).replace(microsecond=0)
    raw_dir = LLM_DIR / 'raw' / stage / f'codex-{model}'
    raw_dir.mkdir(parents=True, exist_ok=True)
    raw_file = f"{item}_{now.strftime('%Y%m%dT%H%M%S')}.json"
    out_file = RUN_CWD / f'{stage}_{item}_{now.strftime("%H%M%S")}.out'
    row = dict(stage=stage, item=item, input_hash=text_hash(text), prompt_hash=prompt_hash, runner='codex', model=model, effort=effort,
               run_utc=now.isoformat(), raw_file=raw_file, status='', error='')
    cmd = ['codex', 'exec', '--json', '--ephemeral', '--skip-git-repo-check', '-s', 'read-only', '--ignore-user-config', '--ignore-rules',
           '-C', str(RUN_CWD), '--output-schema', str(schema_path), '-o', str(out_file), '-c', f'model_reasoning_effort="{effort}"', '-m', model, '-']
    t0 = time.time()
    try:
        proc = subprocess.run(cmd, input=prompt + '\n\n---\n\n# 입력\n\n' + text, capture_output=True, text=True, timeout=3600, cwd=RUN_CWD, env=ENV)
    except subprocess.TimeoutExpired:
        return row | dict(status='timeout', duration_s=round(time.time() - t0, 1))
    row['duration_s'] = round(time.time() - t0, 1)
    events = []
    for line in proc.stdout.splitlines():
        try:
            events.append(json.loads(line))
        except json.JSONDecodeError:
            events.append({'type': 'text', 'text': line})
    last = out_file.read_text(encoding='utf-8') if out_file.exists() else ''
    out_file.unlink(missing_ok=True)
    usage = next((e['usage'] for e in reversed(events) if e.get('type') == 'turn.completed' and isinstance(e.get('usage'), dict)), {})
    errors = [e for e in events if e.get('type') in ('error', 'turn.failed')]
    resp = {'runner': 'codex', 'model': model, 'stage': stage, 'item': item, 'returncode': proc.returncode, 'usage': usage,
            'last_message': last, 'events': events, 'stderr': proc.stderr[-3000:]}
    (raw_dir / raw_file).write_text(json.dumps(resp, ensure_ascii=False, indent=1), encoding='utf-8')
    row.update(input_tokens=usage.get('input_tokens'), cache_read_tokens=usage.get('cached_input_tokens'),
               output_tokens=usage.get('output_tokens'), reasoning_tokens=usage.get('reasoning_output_tokens'))
    if errors or proc.returncode != 0 or not last:
        msg = ' '.join([proc.stderr, json.dumps(errors, ensure_ascii=False)])
        return row | dict(status='limit' if LIMIT_RE.search(msg) else 'error', error=msg.strip()[:300])
    try:
        errs = [e.message for e in jsonschema.Draft202012Validator(json.loads(schema_path.read_text(encoding='utf-8'))).iter_errors(parse_result(resp))]
        row.update(status='ok' if not errs else 'bad_schema', error='; '.join(errs)[:300])
    except (json.JSONDecodeError, TypeError) as e:
        row.update(status='bad_json', error=str(e)[:300])
    return row


base_schema = {'type': 'object', 'additionalProperties': False, 'required': ['items', 'notes'],
               'properties': {'items': {'type': 'array', 'items': {'type': 'object', 'additionalProperties': False, 'required': ['id', 'terms', 'note'],
                                                                   'properties': {'id': {'type': 'string'}, 'terms': {'type': 'array', 'items': {'type': 'string'}},
                                                                                  'note': {'type': 'string'}}}},
                              'notes': {'type': 'string'}}}
BASE_SCHEMA = LLM_DIR / 'schemas' / f'{BASE_PROMPT}.schema.json'
BASE_SCHEMA.parent.mkdir(parents=True, exist_ok=True)
BASE_SCHEMA.write_text(json.dumps(base_schema, ensure_ascii=False, indent=1), encoding='utf-8')
BASE_TEXT = (ROOT / 'prompts' / f'{BASE_PROMPT}.md').read_text(encoding='utf-8') + CODEX_SUFFIX
BASE_HASH = text_hash(BASE_TEXT + BASE_MODEL + BASE_EFFORT)
runs = read_runs()
done = set(runs.loc[(runs['stage'] == BASE_PROMPT) & (runs['status'] == 'ok') & (runs['prompt_hash'] == BASE_HASH), ['item', 'input_hash']].itertuples(index=False, name=None))
todo = {k: t for k, t in BASE_TEXTS.items() if (k, text_hash(t)) not in done}
print(f'이번에 돌릴 것 {len(todo)}건 ({BASE_MODEL}, 추론 {BASE_EFFORT})')
if RUN and todo:
    with ThreadPoolExecutor(3) as pool:
        futs = {pool.submit(run_codex, BASE_PROMPT, k, BASE_TEXT, BASE_SCHEMA, BASE_HASH, t, BASE_MODEL, BASE_EFFORT): k for k, t in todo.items()}
        for f in as_completed(futs):
            row = f.result()
            pd.DataFrame([row]).reindex(columns=RUN_COLUMNS).to_csv(RUNS_PATH, mode='a', header=not RUNS_PATH.exists(), index=False, encoding='utf-8-sig')
            print(f"  {futs[f]:13} {row['status']:8} {row.get('duration_s')}초 입력 {row.get('input_tokens')} 출력 {row.get('output_tokens')} {row.get('error') or ''}")
elif not RUN:
    print('RUN = False: 실행하지 않았다.')

이번에 돌릴 것 0건 (gpt-6.1-sol, 추론 high)
RUN = False: 실행하지 않았다.


In [11]:
# ── 기본 검색어 결과: 형식 점검, 저장, 보기 ──
runs = read_runs()
ok = runs[(runs['stage'] == BASE_PROMPT) & (runs['status'] == 'ok') & (runs['prompt_hash'] == BASE_HASH)].sort_values('run_utc').drop_duplicates('item', keep='last')
BASE_OUT = {}
for r in ok.itertuples():
    if r.item in BASE_TEXTS and r.input_hash == text_hash(BASE_TEXTS[r.item]):
        for x in parse_result(json.loads((LLM_DIR / 'raw' / BASE_PROMPT / f'codex-{BASE_MODEL}' / r.raw_file).read_text(encoding='utf-8')))['items']:
            BASE_OUT[x['id']] = x
if BASE_OUT:
    want = {g for v in BASE_INPUTS.values() for g in v}
    print(f'결과 {len(BASE_OUT)}개(기대 {len(want)}) / 빠진 것 {sorted(want - set(BASE_OUT))} / 없는 ID {sorted(set(BASE_OUT) - want)} / 빈 검색어 {sum(not x["terms"] for x in BASE_OUT.values())}개')
    BASE = pd.DataFrame([dict(id=g, kind=k, name=n, n_members=len(m), terms=' | '.join(BASE_OUT[g]['terms']), note=BASE_OUT[g]['note'])
                         for g, k, n, m in GROUPS if g in BASE_OUT])
    print('base_terms.csv', save_csv(BASE, OUT / 'base_terms.csv'))
    # 이전 버전과 비교: 검색어가 달라진 묶음 수, 비운 것이 바뀐 묶음
    prev_ok = runs[(runs['stage'] == PREV_PROMPT) & (runs['status'] == 'ok')].sort_values('run_utc').drop_duplicates('item', keep='last')
    PREV = {x['id']: x['terms'] for r in prev_ok.itertuples()
            for x in parse_result(json.loads((LLM_DIR / 'raw' / PREV_PROMPT / f'codex-{r.model}' / r.raw_file).read_text(encoding='utf-8')))['items']}
    if PREV:
        changed = [g for g in BASE_OUT if g in PREV and set(map(str.lower, PREV[g])) != set(map(str.lower, BASE_OUT[g]['terms']))]
        print(f'{PREV_PROMPT} → {BASE_PROMPT}: 검색어가 달라진 묶음 {len(changed)}개 / 비운 것 {sorted(g for g in PREV if not PREV[g])} → {sorted(g for g in BASE_OUT if not BASE_OUT[g]["terms"])}')
    pd.set_option('display.max_rows', 200)
    display(BASE[['id', 'n_members', 'terms', 'note']])
else:
    print('아직 결과가 없다.')

결과 139개(기대 139) / 빠진 것 [] / 없는 ID [] / 빈 검색어 7개
base_terms.csv 그대로
search_base_v3 → search_base_v4: 검색어가 달라진 묶음 132개 / 비운 것 ['S:상사,자본재', 'S:소매(유통)', 'T:금융', 'T:부동산신탁', 'T:부동산임대개발', 'T:여가생활'] → ['S:상사,자본재', 'S:소매(유통)', 'T:금융', 'T:부동산신탁', 'T:부동산임대개발', 'T:여가생활', 'T:폐기물처리']


,id,n_members,terms,note
0,S:IT가전,25,EV batteries | home appliances | lithium prices | battery tariffs,가전과 함께 비중이 큰 이차전지 사업을 반영.
1,S:IT하드웨어,53,smartphone shipments | AI servers | PC shipments | electronics tariffs,
2,S:건강관리,115,drug approvals | drug pricing | biosimilar competition | GLP-1 drugs,
3,S:건설,70,Saudi megaprojects | LNG projects | steel prices,해외 플랜트 수주와 국제 자재 가격을 중심으로 선정.
4,S:기계,58,power transformers | grid investment | nuclear power | copper prices | indus...,전력기기 비중을 우선 반영.
5,S:디스플레이,11,OLED panels | LCD prices | display equipment | foldable phones,
6,"S:미디어,교육",22,K-pop | music streaming | streaming competition | advertising spending | box...,미국 뉴스와 연결되는 콘텐츠·공연·광고 중심.
7,S:반도체,85,memory prices | HBM demand | chip equipment | export controls,
8,S:보험,13,Treasury yields | reinsurance rates | insured losses,자산운용 금리와 국제 재보험 시장을 통한 영향을 반영.
9,S:비철금속,50,copper prices | zinc prices | aluminum prices | metal tariffs,제지·포장 제품보다 업종 이름에 맞는 비철금속에 집중.


## 10. 회사 단위 기본 검색어 (Codex)

업종·테마 기본 검색어를 **하나도 못 받는 종목**(업종·테마가 모두 비었거나 중립: 지주사, 상사, 내수 유통 등)은 회사 하나를 입력으로 같은 원칙의 검색어를 만든다.

- 입력 한 줄: 종목코드, 영어 이름, 업종, 테마, 주요 제품(직접 만드는 것 10개), 브랜드(5개), 주요 자회사·투자 대상(지분 관계 5개), 미국 뉴스에 나온 해외 상대·제품.
- 지시문 `prompts/search_company_v2.md`: 9절과 같은 원칙(미국 기사에 그대로 나오는 2단어 중심 표현, 이 회사에 이어지는 경로, 서로 다른 측면, 넓은 말·고유 이름 금지, 중요한 순서). 지주사·상사는 주요 자회사·제품에서 큰 사업 2~3개를 골라 적는다.
- 실행: Codex 구독, `gpt-6.1-sol` + 추론 high, 요청 1건. 결과는 `source/search/company_base_terms.csv`.

| 지시문 | 결과 | 고친 것 |
|---|---|---|
| v1 | 11개 중 10개에 검색어(실리콘투 → K-beauty exports, cosmetics import tariffs / 롯데하이마트 → home appliance prices, memory chip prices / LX홀딩스 → ocean freight rates, display panel demand, methyl methacrylate prices). 신규 상장으로 공시 데이터가 없는 한화머시너리앤서비스홀딩스는 비움. 지역 백화점·할인점은 해외 직구·의류 조달 원가 같은 간접 경로. 평균 2.8단어 | - |
| v2 | 업종·테마 v4가 소매(유통) 등을 채워 대상이 5개로 줄었다. 4개에 모두 2단어 검색어(LX홀딩스 → freight rates, display chips, MMA prices, port congestion / KTcs → call centers, conversational AI), 한화머시너리앤서비스홀딩스는 비움. v2를 쓴다 | 9절 v4와 같은 2단어 규칙 |
| v2 다시 실행(2026-11-17) | 그래프 표기 통일 뒤 9절과 함께 다시 돌림. 업종·테마에서 소매(유통)가 다시 비어 대상이 11개(현대지에프홀딩스, 대구백화점, 한화머시너리앤서비스홀딩스, 광주신세계, KTcs, KTis, 세이브존I&C, 롯데하이마트, 실리콘투, LX홀딩스, 한화갤러리아). 9개에 검색어(실리콘투 → K-beauty, cosmetics tariffs, cosmetics regulation, de minimis / LX홀딩스 → freight rates, coal prices, display chips / 롯데하이마트 → appliance prices, PC shipments), 대구백화점(국내 부동산 임대 중심)과 한화머시너리앤서비스홀딩스는 비움 | - |


In [12]:
# ── 회사 단위 기본 검색어: 업종·테마 기본 검색어가 하나도 없는 종목 → Codex ──
COMPANY_PROMPT = 'search_company_v2'
base_ok = {g for g, x in BASE_OUT.items() if x['terms']}
groups_of = lambda r: ['S:' + r.sector] + ['T:' + t for t in r.themes.split(' | ') if t and THEME_TYPE[t] != '중립']
ORPHANS = [r.stock_code for r in CT.itertuples() if not any(g in base_ok for g in groups_of(r))]


def company_line(code):
    cid = id_of[code]
    r = CT.set_index('stock_code').loc[code]
    g = mine[mine['me'] == cid]
    prods = sorted(set(g.loc[g['type'] == 'PRODUCES', 'other']), key=lambda a: (-p_news.get(a, 0), -num(pd.Series([PRODUCT.at[a, 'n_mentions']]))[0] if a in PRODUCT.index else 0, a))
    brands = [BRAND.at[b, 'name_en'] for b in sorted(set(g.loc[g['type'] == 'HAS_BRAND', 'other']), key=lambda b: (-b_news.get(b, 0), b)) if b in BRAND.index and BRAND.at[b, 'name_en']]
    owns = REL_P[(REL_P['type'] == 'OWNS') & (REL_P['start'] == cid) & REL_P['end'].isin(CO_I.index)]['end']
    subs = sorted(set(owns), key=lambda x: (-co_news.get(x, 0), -degree.get(x, 0), x))
    partners = sorted((x for x in set(g['other']) if x in CO_I.index and co_news.get(x, 0) > 0 and CO_I.at[x, 'country'] not in ('', '대한민국')),
                      key=lambda x: (-co_news[x], x))          # 기사 많은 순(실행마다 같게)
    news_p = [a for a in prods if p_news.get(a, 0) > 0]
    return ' | '.join([code, en_name(cid), r['sector'], r['themes'], ', '.join(p_name(a) for a in prods[:10] if p_name(a)),
                       ', '.join(brands[:5]), ', '.join(en_name(x) for x in subs[:5]), ', '.join(en_name(x) for x in partners[:5]),
                       ', '.join(p_name(a) for a in news_p[:5])])


COMPANY_TEXTS = {'company_orphans': '\n'.join(company_line(c) for c in ORPHANS)} if ORPHANS else {}
COMPANY_TEXT = (ROOT / 'prompts' / f'{COMPANY_PROMPT}.md').read_text(encoding='utf-8') + CODEX_SUFFIX
COMPANY_HASH = text_hash(COMPANY_TEXT + BASE_MODEL + BASE_EFFORT)
print(f'기본 검색어가 하나도 없는 종목 {len(ORPHANS)}개:', ', '.join(CT.set_index("stock_code").loc[ORPHANS, "name"]))
print(COMPANY_TEXTS.get('company_orphans', '')[:1500])
runs = read_runs()
done = set(runs.loc[(runs['stage'] == COMPANY_PROMPT) & (runs['status'] == 'ok') & (runs['prompt_hash'] == COMPANY_HASH), ['item', 'input_hash']].itertuples(index=False, name=None))
todo = {k: t for k, t in COMPANY_TEXTS.items() if (k, text_hash(t)) not in done}
print(f'이번에 돌릴 것 {len(todo)}건 ({BASE_MODEL}, 추론 {BASE_EFFORT})')
if RUN and todo:
    for k, t in todo.items():
        row = run_codex(COMPANY_PROMPT, k, COMPANY_TEXT, BASE_SCHEMA, COMPANY_HASH, t, BASE_MODEL, BASE_EFFORT)
        pd.DataFrame([row]).reindex(columns=RUN_COLUMNS).to_csv(RUNS_PATH, mode='a', header=not RUNS_PATH.exists(), index=False, encoding='utf-8-sig')
        print(f"  {k} {row['status']} {row.get('duration_s')}초 입력 {row.get('input_tokens')} 출력 {row.get('output_tokens')} {row.get('error') or ''}")
elif not RUN:
    print('RUN = False: 실행하지 않았다.')
runs = read_runs()
ok = runs[(runs['stage'] == COMPANY_PROMPT) & (runs['status'] == 'ok') & (runs['prompt_hash'] == COMPANY_HASH)].sort_values('run_utc').drop_duplicates('item', keep='last')
COMPANY_OUT = {}
for r in ok.itertuples():
    if r.item in COMPANY_TEXTS and r.input_hash == text_hash(COMPANY_TEXTS[r.item]):
        for x in parse_result(json.loads((LLM_DIR / 'raw' / COMPANY_PROMPT / f'codex-{BASE_MODEL}' / r.raw_file).read_text(encoding='utf-8')))['items']:
            COMPANY_OUT[x['id']] = x
if COMPANY_OUT:
    print(f'결과 {len(COMPANY_OUT)}개(기대 {len(ORPHANS)}) / 빠진 것 {sorted(set(ORPHANS) - set(COMPANY_OUT))} / 빈 검색어 {sum(not x["terms"] for x in COMPANY_OUT.values())}개')
    CB = pd.DataFrame([dict(stock_code=c, name=CT.set_index('stock_code').at[c, 'name'], terms=' | '.join(COMPANY_OUT[c]['terms']), note=COMPANY_OUT[c]['note'])
                       for c in ORPHANS if c in COMPANY_OUT])
    print('company_base_terms.csv', save_csv(CB, OUT / 'company_base_terms.csv'))
    display(CB)
else:
    print('아직 결과가 없다.')

기본 검색어가 하나도 없는 종목 11개: 현대지에프홀딩스, 대구백화점, 한화머시너리앤서비스홀딩스, 광주신세계, KTcs, KTis, 세이브존I&C, 롯데하이마트, 실리콘투, LX홀딩스, 한화갤러리아
005440 | HYUNDAI G.F | 상사,자본재 | 유통 | plastic resins, apparel, food ingredients, pharmaceuticals, software, travel and tourism services, construction and heavy equipment, contract food service, airline ticket booking and ticketing, investment property leasing | DKNY, Club Monaco, Lanvin, Lanvin Blanc, MUEden | Hyundai Motor, Hyundai Livart, Hyundai Department Store, Daewon Kang Up, Hyundai Futurenet |  | plastic resins
006370 | Daegu Department Store | 소매(유통) | 유통 | 부동산임대개발 | equipment rental and leasing, apparel, general merchandise |  | (주)그린바이오텍, 기업구조조정 1호, (주)대구시민프로축구단, (주)대백아울렛, (주)엑스코 |  | 
0220W0 | Hanwha Machinery & Service | 상사,자본재 | 지주사 |  |  |  |  | 
037710 | GwangjuShinsegae | 소매(유통) | 유통 | real estate leasing, apparel, express bus parcel delivery, passenger terminal operations, passenger ticketing |  | 신세계의정부역사 |  | 
058850 | ktcs | 상사,자본재 | 컨택센터 | 유통 | AI solution

,stock_code,name,terms,note
0,005440,현대지에프홀딩스,beef prices | luxury spending | resin prices | Chinese tourists,"사업별 매출 비중이 없어 급식·식자재, 백화점, 합성수지 사업에 나눠 반영."
1,006370,대구백화점,,"주요 제품이 국내 부동산 임대·개발로 기재돼 있으며, 미국·세계 시장으로 이어지는 사업 경로가 확인되지 않음."
2,0220W0,한화머시너리앤서비스홀딩스,,제품·자회사 정보가 없어 이름만으로 기계 사업이나 특정 산업 노출을 추정하지 않음.
3,037710,광주신세계,luxury spending | Chinese tourists | cotton prices,글로벌 명품 수요와 의류 원가를 반영. 지역 점포의 해외 관광객 노출은 제한적일 수 있음.
4,058850,KTcs,call centers | voice AI | smartphone prices,컨택센터 자동화와 휴대전화 유통을 반영. 미국 통신 가입 지표는 국내 사업과 연결이 약함.
5,058860,KTis,call centers | voice AI | offshore outsourcing,해외 아웃소싱은 국내 매출보다 세계 컨택센터 산업의 경쟁·운영 방식 변화와 연결됨.
6,067830,세이브존I&C,fast fashion | cotton prices | apparel sourcing,글로벌 의류 경쟁과 조달 원가를 통한 간접 영향 위주이며 해외 매출은 확인되지 않음.
7,071840,롯데하이마트,appliance prices | PC shipments | smartphone prices,국내 판매에 영향을 주는 글로벌 제조사 가격·제품 공급 변화를 반영.
8,257720,실리콘투,K-beauty | cosmetics tariffs | cosmetics regulation | de minimis,소액면세 변화는 직배송 물량에 주로 영향을 주며 현지 도매·물류 사업에는 영향이 다를 수 있음.
9,383800,LX홀딩스,freight rates | coal prices | display chips,LX International의 물류·자원 사업과 LX Semicon의 디스플레이 반도체 사업을 우선 반영.


## 11. 구글 뉴스 RSS 시험 목록

시험 종목 18개: 유형마다 대형·중소형, 업종과 테마가 다른 사례(크래프톤, 우진), 연결이 적은 종목(농심, 아모레퍼시픽, 하이브, GKL, 티에프이), 업종·테마 기본 검색어가 없어 회사 기본 검색어를 받는 종목(실리콘투, LX홀딩스).

검색어마다 **모양**을 정해 검색 문자열을 만든다(구문은 큰따옴표로 묶는다).

| 종류 | 모양 |
|---|---|
| 자기 회사 | 이름만 |
| 구체·넓은 제품, 원자재·지표, 정책, 테마 대표 | 주제만 |
| 브랜드 | 이름만 |
| 해외 상대 | 이름만 / 상대 + 관계 제품(관계에 제품이 있을 때) |
| 업종·테마 기본(9절) | 주제만(업종 2개, 테마는 중립 제외 앞의 2개에서 2개씩) |
| 회사 기본(10절) | 주제만(앞의 2개) |

- 2차 시험(`news_search_tests/search_tests.ipynb` 6~8절)에서 바꾼 것: "회사 + 주제"·"회사 + 브랜드" 모양은 88%가 0건이라 뺐다.
- 3차 시험(`news_search_tests/search_tests.ipynb` 9절)에서 3단어 이상 주제의 **따옴표 없는 모양**(단어가 모두 들어간 기사)을 함께 시험했다. 기사 수는 중앙값 0 → 42건으로 늘었지만 제목을 보면 대부분 상관없는 기사였다(research and development services 565건: 양자컴퓨터 종목 추천, 비트코인 등. 긴 금융 기사에는 흔한 단어가 다 들어 있다). 그래서 쓰지 않고, 긴 검색어는 짧게 줄이는 쪽으로 간다(`LONG_WORDS = None`이면 만들지 않는다. 시험할 때만 3으로).
- 이 표는 지금 규칙으로 만든 최신 목록이다. 시험마다 쓴 목록은 `news_search/source/<시험 폴더>/test_queries.csv`에 고정해 둔다(규칙이 바뀌어도 끝난 시험은 그 목록을 다시 읽는다).

같은 검색 문자열은 한 번만 검색한다(테마 대표처럼 여러 종목이 함께 쓰는 것). 검색 실행과 결과 분석은 `news_search_tests/search_tests.ipynb`가 이 표(`test_queries.csv`)를 읽어서 한다.

In [13]:
# ── 시험 목록: 시험 종목의 검색어 후보 → 모양별 검색 문자열(중복 없이) ──
TEST_COMPANIES = ['SK하이닉스', '한미반도체', '티에프이', '셀트리온', '펩트론', '농심', '아모레퍼시픽', '크래프톤', '하이브', 'GKL',
                  'HD한국조선해양', '한화에어로스페이스', '우진', 'POSCO홀딩스', '대한항공', 'KB금융',
                  '실리콘투', 'LX홀딩스']                      # 업종·테마 기본 검색어가 없어 회사 기본 검색어(10절)를 받는 종목
gone = [n for n in TEST_COMPANIES if n not in set(CT['name'])]   # 시험 종목은 2026-11 데이터에서 고른 목록이다
if gone:
    print(f'※ 시험 종목 {len(gone)}개는 이번 대상에 없어 뺀다(다시 실행해 데이터가 달라지면 시험 종목을 다시 고르기를 권한다):', gone)
TEST_COMPANIES = [n for n in TEST_COMPANIES if n not in gone]
quote = lambda s: f'"{s}"'
LONG_WORDS = None                                 # 숫자(예: 3)면 그 단어 수 이상인 주제에 따옴표 없는 모양도 만든다(3차 시험: 상관없는 기사가 대부분이라 쓰지 않음)
BASE_PER_GROUP = 2                                # 업종·테마·회사 기본 검색어를 묶음마다 앞에서 몇 개 쓰나
THEMES_PER_COMPANY = 2                            # 테마 기본 검색어를 종목의 테마(중립 제외) 앞에서 몇 개 테마까지 쓰나


def topic_shapes(term):
    out = [('주제만', quote(term))]
    if LONG_WORDS and len(term.split()) >= LONG_WORDS:
        out.append(('주제만·따옴표 없음', term))
    return out


def company_query_rows(codes):
    """종목코드 목록 → 검색 문자열 줄 목록. 11절 시험 목록과 12절 전체 표가 같은 규칙으로 쓴다.
    한 줄: 종목코드, 종목명, 기본 유형, 연결이 적은지, 검색어 종류, 검색어, 업종·테마 이름, 모양, 검색 문자열, 근거 노드, 미국 뉴스 기사 수.
    순서: ① 모든 종목의 그래프 검색어 후보(8절) ② 업종·테마 기본 검색어(9절) ③ 회사 기본 검색어(10절)."""
    info = CT.set_index('stock_code')
    rows = []
    for code in codes:                               # ① 그래프 검색어 후보: 자기 회사·해외 상대·브랜드는 이름만, 나머지는 주제만
        r = info.loc[code]
        for kind, term, extra, node, n in candidates(id_of[code]):
            if not term:
                continue
            shapes = [('이름만', quote(term))] if kind in ('자기 회사', '브랜드', '해외 상대') else topic_shapes(term)
            if kind == '해외 상대' and extra:
                shapes.append(('상대 + 제품', f'{quote(term)} {quote(extra)}'))
            for shape, q in shapes:
                rows.append(dict(stock_code=code, company=r['name'], base_type=r['base_type'], weak=r['weak'], kind=kind, term=term,
                                 theme=extra if kind == '테마 대표' else '', shape=shape, query=q, node=node, n_news=n))
    base_terms = {g: [t for t in x['terms']] for g, x in BASE_OUT.items()} if BASE_OUT else {}
    for code in codes:                               # ② 업종·테마 기본 검색어: 종목의 업종과 테마(중립 제외 앞의 2개)에서 2개씩, 주제만
        r = info.loc[code]
        groups = [('업종 기본', 'S:' + r['sector'])] + [('테마 기본', 'T:' + t) for t in [t for t in r['themes'].split(' | ') if t and THEME_TYPE[t] != '중립'][:THEMES_PER_COMPANY]]
        for kind, g in groups:
            for term in base_terms.get(g, [])[:BASE_PER_GROUP]:
                for shape, q in topic_shapes(term):
                    rows.append(dict(stock_code=code, company=r['name'], base_type=r['base_type'], weak=r['weak'], kind=kind, term=term, theme=g[2:],
                                     shape=shape, query=q, node=g, n_news=0))
    for code in codes:                               # ③ 회사 기본 검색어: 앞의 2개, 주제만
        r = info.loc[code]
        for term in (COMPANY_OUT.get(code, {}).get('terms') or [])[:BASE_PER_GROUP]:
            for shape, q in topic_shapes(term):
                rows.append(dict(stock_code=code, company=r['name'], base_type=r['base_type'], weak=r['weak'], kind='회사 기본', term=term, theme='',
                                 shape=shape, query=q, node='KR:' + code, n_news=0))
    return rows


code_of_name = CT.set_index('name')['stock_code']
TESTQ = pd.DataFrame(company_query_rows([code_of_name[n] for n in TEST_COMPANIES])).drop(columns=['stock_code'])   # 시험 목록은 예전 칸 그대로
print(f"시험 종목 {TESTQ['company'].nunique()}개 / 검색어 후보 {TESTQ[['company', 'kind', 'term']].drop_duplicates().shape[0]}개 / "
      f"검색 문자열 {len(TESTQ)}줄, 중복 없이 {TESTQ['query'].nunique()}개")
print('종류별 검색 문자열', TESTQ.groupby('kind')['query'].nunique().to_dict())
print('모양별 검색 문자열', TESTQ.groupby('shape')['query'].nunique().to_dict())
print('save', save_csv(TESTQ, OUT / 'test_queries.csv'))
pd.set_option('display.max_rows', 400)
display(TESTQ[['company', 'kind', 'shape', 'query', 'n_news']])

시험 종목 18개 / 검색어 후보 217개 / 검색 문자열 222줄, 중복 없이 170개
종류별 검색 문자열 {'구체 제품': 40, '넓은 제품': 14, '브랜드': 17, '업종 기본': 24, '원자재·지표': 7, '자기 회사': 19, '정책': 2, '테마 기본': 39, '테마 대표': 13, '해외 상대': 15, '회사 기본': 4}
모양별 검색 문자열 {'상대 + 제품': 4, '이름만': 47, '주제만': 119}
save 그대로


,company,kind,shape,query,n_news
0,SK하이닉스,자기 회사,이름만,"""SK hynix""",15
1,SK하이닉스,구체 제품,주제만,"""high-bandwidth memory""",9
2,SK하이닉스,구체 제품,주제만,"""semiconductor chips""",9
3,SK하이닉스,구체 제품,주제만,"""random-access memory""",2
4,SK하이닉스,넓은 제품,주제만,"""memory chips""",15
5,SK하이닉스,해외 상대,이름만,"""Nvidia""",146
6,SK하이닉스,해외 상대,상대 + 제품,"""Nvidia"" ""memory chips""",146
7,SK하이닉스,해외 상대,이름만,"""Kioxia""",4
8,SK하이닉스,원자재·지표,주제만,"""semiconductor wafers""",1
9,SK하이닉스,테마 대표,주제만,"""semiconductor testing services""",1


## 12. 전체 검색어 표

서비스 대상 후보 **1,067개 전체**에 11절과 같은 규칙(`company_query_rows`)을 적용해 검색어 표를 만든다. 밤사이 수집(`news_search/news_collect.ipynb`)은 나중에 이 표에서 보유 종목(수집 종목)의 검색어만 골라 쓴다. 지금 수집은 아직 시험 목록 사본(`news_search/source/terms/`)을 쓰고, 검색어를 줄이는 규칙(다른 뜻으로 걸리는 이름, 늘 0건인 검색어, 종목당 수)을 정한 뒤 이 표로 바꾼다(`graphDB.md` 결정할 것 #22).

**출력** `source/search/queries_all.csv`(내용이 바뀔 때만 쓴다): 한 줄 = 종목 하나의 검색 문자열 하나. 칸은 11절 시험 목록과 같고 `stock_code`(종목코드)를 더했다. 여러 종목이 같은 검색 문자열을 쓰면 종목마다 한 줄씩 있고, 검색은 검색 문자열마다 한 번만 한다.

| 칸 | 내용 |
|---|---|
| `stock_code`, `company` | 종목코드, 종목명 |
| `base_type`, `weak` | 종목 유형(4절), 연결이 적은 종목인지(5절) |
| `kind`, `term` | 검색어 종류(자기 회사, 구체·넓은 제품, 브랜드, 해외 상대, 원자재·지표, 정책, 테마 대표, 업종·테마·회사 기본)와 검색어 |
| `theme` | 테마 대표·업종·테마 기본이면 그 업종·테마 이름 |
| `shape`, `query` | 모양(이름만, 주제만, 상대 + 제품)과 구글 뉴스 RSS에 넣는 검색 문자열 |
| `node`, `n_news` | 근거 그래프 노드(그래프 연결 근거를 만들 때 쓴다), 그 노드의 미국 뉴스 기사 수 |

**함께 보는 숫자**(검색어를 줄이는 규칙을 정할 재료): 검색 문자열 수(중복 없이), 종목당 검색 문자열 수, 종류별 수, 여러 종목이 같이 쓰는 검색 문자열, 지금 요청 간격(6초)으로 전체를 한 번 도는 데 걸리는 시간과 하루 요청 수.

In [14]:
# ── 전체 검색어 표: 서비스 대상 후보 전체 → queries_all.csv, 규모 보기 ──
SEC_PER_REQUEST = 6.4        # 요청 1건에 걸리는 시간(초): news_search의 요청 간격 6초 + 응답 시간(2026-11-16 실측)
ROUNDS_PER_DAY = 14         # news_search 하루 회차 수(2026-11-17 일정): 밤 22:00~수집 마감 1시간 간격 9번(마감 회차 포함), 저녁 16~22시 2시간 간격 3번, 낮 4시간 간격 2번

ALLQ = pd.DataFrame(company_query_rows(list(CT['stock_code'])))
print('queries_all.csv', save_csv(ALLQ, OUT / 'queries_all.csv'))
uq = ALLQ.drop_duplicates('query')
per = ALLQ.groupby('stock_code')['query'].nunique()
print(f"종목 {ALLQ['stock_code'].nunique()}개(대상 {len(CT)}개) / 검색 문자열 {len(ALLQ):,}줄, 중복 없이 {len(uq):,}개")
print(f"종목당 검색 문자열: 최소 {per.min()}, 중앙값 {per.median():.0f}, 최대 {per.max()} / 검색어가 없는 종목 {len(CT) - len(per)}개")

# 종류별: 검색 문자열 수(중복 없이), 같이 쓰는 종목 수
share = ALLQ.groupby('query')['stock_code'].nunique()
kinds = ALLQ.drop_duplicates(['query', 'kind']).assign(n_companies=lambda d: d['query'].map(share))
print('\n종류별 검색 문자열(중복 없이)과 한 검색 문자열을 같이 쓰는 종목 수')
display(kinds.groupby('kind').agg(검색문자열=('query', 'nunique'), 같이쓰는_종목_평균=('n_companies', 'mean'), 같이쓰는_종목_최대=('n_companies', 'max'),
                                  미국뉴스_0건=('n_news', lambda s: int((s == 0).sum()))).round(1).sort_values('검색문자열', ascending=False))

# 여러 종목이 같이 쓰는 검색 문자열(해외 상대·테마는 한 번 검색해 여러 종목이 나눠 쓴다)
top = ALLQ.groupby(['query', 'kind'])['stock_code'].nunique().rename('종목수').reset_index().sort_values('종목수', ascending=False)
print('\n여러 종목이 같이 쓰는 검색 문자열 상위')
display(top.head(25))
print('해외 상대 상위: 몇 종목이 같은 해외 회사를 검색어로 쓰나')
display(top[top['kind'] == '해외 상대'].head(15))

# 요청 규모: 검색 문자열마다 3건(야후 + 5곳 묶음 2), 6초 간격
round_h = len(uq) * 3 * SEC_PER_REQUEST / 3600
max_per_day = int(24 * 3600 / SEC_PER_REQUEST)
need = len(uq) * 3 * ROUNDS_PER_DAY
print(f"\n전체를 한 번 도는 회차: 요청 {len(uq) * 3:,}건, 약 {round_h:.1f}시간(지금 시험 목록 170개는 약 55분)")
print(f"지금 일정(하루 {ROUNDS_PER_DAY}회차)대로 하루 요청: {need:,}건 / 6초 간격으로 하루에 보낼 수 있는 최대 {max_per_day:,}건")
print(f"→ 밤 1시간 간격을 지키려면 한 회차가 1시간 안이어야 해서, 검색 문자열은 약 {int(3600 / SEC_PER_REQUEST / 3):,}개까지 된다")

queries_all.csv 그대로
종목 1067개(대상 1067개) / 검색 문자열 12,695줄, 중복 없이 4,145개
종목당 검색 문자열: 최소 1, 중앙값 11, 최대 21 / 검색어가 없는 종목 0개

종류별 검색 문자열(중복 없이)과 한 검색 문자열을 같이 쓰는 종목 수


,검색문자열,같이쓰는_종목_평균,같이쓰는_종목_최대,미국뉴스_0건
kind,,,,
구체 제품,1821,2.5,118,1595
자기 회사,1182,1.0,3,1129
브랜드,697,1.1,3,661
테마 기본,197,19.5,119,197
해외 상대,114,2.6,13,0
원자재·지표,105,13.5,108,0
넓은 제품,75,9.9,98,56
테마 대표,72,32.6,118,0
업종 기본,46,53.5,119,46



여러 종목이 같이 쓰는 검색 문자열 상위


,query,kind,종목수
3208,"""drug pricing""",업종 기본,115
3205,"""drug approvals""",업종 기본,115
3849,"""oil tankers""",테마 대표,109
3553,"""hybrid electric vehicles""",테마 대표,97
1015,"""HBM demand""",테마 기본,97
3750,"""memory chips""",테마 기본,97
2989,"""container ships""",테마 대표,94
2451,"""auto tariffs""",테마 기본,93
4304,"""vehicle sales""",테마 기본,93
4070,"""semiconductor chips""",테마 대표,92


해외 상대 상위: 몇 종목이 같은 해외 회사를 검색어로 쓰나


,query,kind,종목수
846,"""Eli Lilly""",해외 상대,13
955,"""General Motors""",해외 상대,13
1660,"""Novartis""",해외 상대,13
938,"""GSK(GlaxoSmithKline)""",해외 상대,12
2164,"""Volkswagen""",해외 상대,11
1807,"""Roche""",해외 상대,9
2022,"""Siemens""",해외 상대,9
1571,"""Microsoft""",해외 상대,9
1745,"""Panasonic""",해외 상대,8
905,"""Ford""",해외 상대,8



전체를 한 번 도는 회차: 요청 12,435건, 약 22.1시간(지금 시험 목록 170개는 약 55분)
지금 일정(하루 14회차)대로 하루 요청: 174,090건 / 6초 간격으로 하루에 보낼 수 있는 최대 13,500건
→ 밤 1시간 간격을 지키려면 한 회차가 1시간 안이어야 해서, 검색 문자열은 약 187개까지 된다
